# Layer 5-7: Backtest Deep Dive Template

This notebook demonstrates the standardized workflow for running the `VectorizedBacktester` and generating institutional reports using the `QuantReporter` with **ADR-010 Traceability**.

In [ ]:
import os
import sys
import pandas as pd

# Add project root to path
sys.path.append('..')

from scripts.libs.data.loader import FrameworkLoader
from scripts.trading_framework.strategies.logic.box_reversion import BoxMeanReversionSignal
from scripts.trading_framework.core.backtest_engine import VectorizedBacktester
from scripts.trading_framework.reporting.reporter import QuantReporter

## 1. Load Data (Layer 1)

In [ ]:
ticker = 'NQ1'
loader = FrameworkLoader(ticker=ticker)
df = loader.load(include_historical=True)
df.tail()

## 2. Generate Signals (Layer 4)

In [ ]:
strategy = BoxMeanReversionSignal()
config = {
    'filter_high_vol': True,
    'min_dist': 0.0015,
    'tp_buffer': 0.0002,
    'sl_dist': 0.0050,
    'strategy_name': 'Manual_DeepDive'
}

signals = strategy.generate_signals(df.last('30D'), config)
signals.value_counts()

## 3. Run Vectorized Engine (Layer 5)

In [ ]:
engine = VectorizedBacktester()
metrics = engine.run(signals, df.last('30D'))
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Max drawdown: {metrics['max_drawdown_%']:.2f}%")

## 4. Institutional Reporting (Layer 7 + ADR-010)

In [ ]:
from datetime import datetime
run_id = f"RESEARCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{ticker}_DeepDive"
reporter = QuantReporter(run_id=run_id)

returns = metrics['equity_curve'].pct_change().fillna(0)
reporter.generate_tear_sheet(returns, "Research_TearSheet")
print(f"Results stored in: {reporter.output_dir}")